In [1]:
import sys
import os
import pandas as pd
import numpy as np
import csv
import pip
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import statsmodels.api as sm
import linearmodels as lm
from linearmodels import PanelOLS, RandomEffects
from scipy import stats
from linearmodels import RandomEffects
import statsmodels.api as sm
from linearmodels.panel import compare

In [2]:
final_df = pd.read_csv("prepared_data_for_regression.csv")
print(final_df.head(5))

      country  year  incidence  mortality       log_gdp       bmi    pop_65  \
0        fiji  2000  39.310026  33.722967   9426.113813  0.303726  3.437768   
1    cambodia  2000  11.807551  10.664151   1922.031791  0.100004  2.816518   
2    kiribati  2000  13.866566  12.601468   2481.415243  0.307055  3.413156   
3  kazakhstan  2000  35.284421  18.655098  12935.864368  0.291659  6.693366   
4     jamaica  2000  50.404926  25.599975   9518.231763  0.240683  6.060642   

   urban_pop  fertility  labor_rate  internet  health_exp  smoking  hosp_beds  \
0     47.908      2.992      37.841  1.496850    3.424412     15.9       2.05   
1     18.586      3.794      77.834  0.047023    6.482484     23.5       0.60   
2     42.958      4.071         NaN  1.785230    7.773798     53.9       1.32   
3     56.098      1.898      65.381  0.668594    4.160324     12.0       6.90   
4     51.814      2.345      58.147  3.115780    5.644625      8.8       1.70   

        MIR  
0  0.857872  
1  0.90316

In [3]:
initial_countries=final_df['country'].nunique()
print(initial_countries)

166


In [4]:
initial_years=final_df['year'].nunique()
print(initial_years)

24


# OECD  Countries :

In [5]:
oecd_countries = ['austria', 'australia', 'belgium', 'canada', 'chile', 
                  'colombia', 'czechia', 'denmark', 'estonia', 'finland', 
                  'france', 'germany', 'greece', 'hungary', 'iceland', 'ireland', 
                  'israel', 'italy', 'japan', 'korea', 'latvia', 'lithuania', 'luxembourg', 
                  'mexico', 'netherlands', 'new zealand', 'norway', 'poland', 'portugal', 'slovakia', 
                  'slovenia', 'spain', 'sweden', 'switzerland', 'turkiye', 'united kingdom', 'united states', 'costa rica']
print(len(oecd_countries))

38


# Create Dummy Variables:

In [6]:
final_df['is_oecd'] = final_df['country'].isin(oecd_countries).astype(int)

# create interaction :

In [7]:
final_df['bmi_x_oecd'] = final_df['bmi']* final_df['is_oecd']
print(final_df['is_oecd'].value_counts())

is_oecd
0    2860
1     808
Name: count, dtype: int64


# Dummy Variables: 

In [8]:
oecd_comparison= final_df.groupby('is_oecd').incidence.mean()
print(final_df['is_oecd'].value_counts())

is_oecd
0    2860
1     808
Name: count, dtype: int64


1. Aging Society Dummy: Based on WHO standards, societies with >10% population over 65 are considered "ageing" or "aged". This captures potential non-linear increases in cancer incidence due to population structure.

In [9]:
final_df['dm_high_aging_society'] = (final_df['pop_65'] > 10).astype(int)
print(final_df['dm_high_aging_society'].value_counts())

dm_high_aging_society
0    2457
1    1211
Name: count, dtype: int64


2. High Life Expectancy Dummy threshold of 75 years represents advanced healthcare systems and higher probability of disease detection.

In [10]:
print(final_df[['year', 'country', 'is_oecd', 'dm_high_aging_society']].head(100))

    year       country  is_oecd  dm_high_aging_society
0   2000          fiji        0                      0
1   2000      cambodia        0                      0
2   2000      kiribati        0                      0
3   2000    kazakhstan        0                      0
4   2000       jamaica        0                      0
..   ...           ...      ...                    ...
95  2000       denmark        1                      1
96  2001          oman        0                      0
97  2000       lebanon        0                      0
98  2000       namibia        0                      0
99  2000  saudi arabia        0                      0

[100 rows x 4 columns]


# interaction of is_oecd variable and gdp

In [11]:
final_df['log_gdp_is_oecd']= final_df['log_gdp']*final_df['is_oecd']
print(final_df['log_gdp_is_oecd'].head(5))

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: log_gdp_is_oecd, dtype: float64


In [12]:
final_df['is_oecd'] = 0
final_df.loc[final_df['country'].isin(oecd_countries), 'is_oecd'] = 1

# interaction of is_oecd variable and gdp
final_df['log_gdp_is_oecd'] = final_df['log_gdp'] * final_df['is_oecd']
# Filling NaNs in Interaction Term: Since 'NaN * 0 = NaN' in Python, rows where is_oecd is 0 but log_gdp is missing(NAN) 
# would incorrectly stay as NaN (and be counted in the results), then filled these with 0 to ensure only real OECD countries with data are counted.
final_df['log_gdp_is_oecd'] = final_df['log_gdp_is_oecd'].fillna(0)

# counting contries :
oecd_count = final_df[final_df['is_oecd'] == 1]['country'].nunique()
interaction_countries = final_df[final_df['log_gdp_is_oecd'] != 0]['country'].nunique()

In [13]:
#print(final_df[['year', 'country', 'gdp_is_oecd', 'dm_high_aging_society', 'dm_high_life_exp']].tail(100))
print(final_df[['country','log_gdp','is_oecd','log_gdp_is_oecd','dm_high_aging_society']].sample(15).round(2))

                     country    log_gdp  is_oecd  log_gdp_is_oecd  \
530                  myanmar    2110.24        0             0.00   
1830              luxembourg  129945.45        1        129945.45   
1058                   china    8013.72        0             0.00   
2505  bosnia and herzegovina   16349.07        0             0.00   
2213                  sweden   58975.98        1         58975.98   
213        brunei darussalam  101571.58        0             0.00   
1116                    togo    1863.28        0             0.00   
1137               indonesia    7140.85        0             0.00   
3229                    mali    2759.11        0             0.00   
528                 botswana   13984.84        0             0.00   
191                 barbados   18380.26        0             0.00   
866                  andorra   65534.12        0             0.00   
1099                   haiti    3289.71        0             0.00   
2059                bulgaria   245

# delete percentage of rows with missing value(NA):

In [31]:
missing_report = ((final_df.isnull().sum() / len(final_df)) * 100).round(2)
print(missing_report[missing_report > 0]) 

log_gdp        1.09
labor_rate     2.48
internet       1.64
smoking       10.88
hosp_beds      0.63
dtype: float64


In [15]:
clean_df= final_df.dropna(subset=['mortality', 'log_gdp', 'pop_65', 'urban_pop', 'fertility', 'labor_rate', 'log_gdp_is_oecd', 'internet', 'health_exp', 'smoking'])
print(f" Missing values after deleting NA in clean_df: {clean_df.isnull().sum()}")

 Missing values after deleting NA in clean_df: country                   0
year                      0
incidence                 0
mortality                 0
log_gdp                   0
bmi                       0
pop_65                    0
urban_pop                 0
fertility                 0
labor_rate                0
internet                  0
health_exp                0
smoking                   0
hosp_beds                23
MIR                       0
is_oecd                   0
bmi_x_oecd                0
dm_high_aging_society     0
log_gdp_is_oecd           0
dtype: int64


# Building stepwise regression

# Model 1: Adding gdp and urbanisation:

In [32]:
df_step= clean_df.set_index(['country', 'year'])
exog_1_fe= sm.add_constant(df_step[['log_gdp', 'urban_pop']])
model_1_fe=PanelOLS(df_step['mortality'],exog_1_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered',
                                                                                                 cluster_entity=True)
print(model_1_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.1591
Estimator:                   PanelOLS   R-squared (Between):             -0.8698
No. Observations:                3147   R-squared (Within):               0.1613
Date:                Fri, Mar 27 2026   R-squared (Overall):             -0.7037
Time:                        17:27:36   Log-likelihood                   -7047.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      282.48
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(2,2985)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             16.001
                            

# Model 2: Adding Pop_65:

In [34]:
exog_2_fe=sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65']])
model_2_fe = PanelOLS(df_step['mortality'], exog_2_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered',
                                                                                                   cluster_entity=True)
print(model_2_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.2858
Estimator:                   PanelOLS   R-squared (Between):             -1.5765
No. Observations:                3147   R-squared (Within):               0.1853
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.3086
Time:                        17:31:05   Log-likelihood                   -6790.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      398.10
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(3,2984)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             21.812
                            

# Model 3: Adding Labor Rate:

In [35]:
exog_3_fe=sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate']])
model_3_fe=PanelOLS(df_step['mortality'], exog_3_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity= True)
print(model_3_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.3080
Estimator:                   PanelOLS   R-squared (Between):             -1.7571
No. Observations:                3147   R-squared (Within):               0.2108
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.4603
Time:                        17:31:52   Log-likelihood                   -6740.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      331.97
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(4,2983)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             23.085
                            

# Model 4: Adding BMI:

In [36]:
exog_4_fe=sm.add_constant(df_step[['log_gdp','urban_pop', 'pop_65', 'labor_rate', 'bmi']])
model_4_fe=PanelOLS(df_step['mortality'], exog_4_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                   cluster_entity= True)
print(model_4_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.3635
Estimator:                   PanelOLS   R-squared (Between):             -1.6644
No. Observations:                3147   R-squared (Within):               0.2496
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.3665
Time:                        17:32:32   Log-likelihood                   -6608.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      340.66
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(5,2982)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             25.380
                            

# Model 5:Adding is_oecd as Dummy Variable:

In [37]:
exog_5_fe=sm.add_constant(df_step[['log_gdp','urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd']])
model_5_fe=PanelOLS(df_step['mortality'], exog_5_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered',
                                                                                                  cluster_entity=True)
print(model_5_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.3760
Estimator:                   PanelOLS   R-squared (Between):             -2.1214
No. Observations:                3147   R-squared (Within):               0.2572
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.7699
Time:                        17:33:02   Log-likelihood                   -6577.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      299.37
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(6,2981)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             21.496
                            

# Model 6: Adding High aging society as Dummy Variable:

In [40]:
exog_6_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd', 'dm_high_aging_society']])
model_6_fe = PanelOLS(df_step['mortality'],exog_6_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered', 
                                                                                                  cluster_entity= True)
print(model_6_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.3806
Estimator:                   PanelOLS   R-squared (Between):             -2.3165
No. Observations:                3147   R-squared (Within):               0.2690
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.9355
Time:                        17:37:10   Log-likelihood                   -6566.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      261.60
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(7,2980)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             18.790
                            

# create labor_rate * is_oecd variable:

In [42]:
df_step['labor_rate * is_oecd']= df_step['labor_rate']* df_step['is_oecd']

# Model 7: Adding labor_rate * is_oecd:

In [44]:
exog_7_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd', 'dm_high_aging_society',
                                    'labor_rate * is_oecd']])
model_7_fe = PanelOLS(df_step['mortality'], exog_7_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                    cluster_entity=True)
print(model_7_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.3835
Estimator:                   PanelOLS   R-squared (Between):             -3.2385
No. Observations:                3147   R-squared (Within):               0.2633
Date:                Fri, Mar 27 2026   R-squared (Overall):             -2.7414
Time:                        17:38:46   Log-likelihood                   -6558.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      231.68
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(8,2979)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             17.241
                            

# Model 8: Adding internet:

In [45]:
exog_8_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd',
                                     'dm_high_aging_society', 'labor_rate * is_oecd','internet']])
model_8_fe = PanelOLS(df_step['mortality'], exog_8_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                     cluster_entity=True)
print(model_8_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.3922
Estimator:                   PanelOLS   R-squared (Between):             -3.4940
No. Observations:                3147   R-squared (Within):               0.1230
Date:                Fri, Mar 27 2026   R-squared (Overall):             -2.9880
Time:                        17:39:46   Log-likelihood                   -6536.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      213.48
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(9,2978)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             15.657
                            

# Model  9: Adding smoking:

In [46]:
exog_9_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd',
                                     'dm_high_aging_society', 'labor_rate * is_oecd','internet',
                                       'smoking']])
model_9_fe = PanelOLS(df_step['mortality'], exog_9_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                    cluster_entity=True)
print(model_9_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.4140
Estimator:                   PanelOLS   R-squared (Between):             -2.7662
No. Observations:                3147   R-squared (Within):              -0.0727
Date:                Fri, Mar 27 2026   R-squared (Overall):             -2.3914
Time:                        17:40:25   Log-likelihood                   -6478.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      210.30
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(10,2977)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             15.442
                            

# Model 10: adding health_exp:

In [47]:
exog_10_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd','dm_high_aging_society'
                                      , 'labor_rate * is_oecd','internet','smoking','health_exp']])
model_10_fe = PanelOLS(df_step['mortality'], exog_10_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                      cluster_entity=True)
print(model_10_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.4182
Estimator:                   PanelOLS   R-squared (Between):             -2.8640
No. Observations:                3147   R-squared (Within):              -0.1222
Date:                Fri, Mar 27 2026   R-squared (Overall):             -2.4846
Time:                        17:41:01   Log-likelihood                   -6467.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      194.44
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(11,2976)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             14.982
                            

# Model 11: adding Fertality variable:

In [49]:
exog_11_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd','dm_high_aging_society',
                                      'labor_rate * is_oecd','internet','smoking','health_exp','fertility']])
model_11_fe = PanelOLS(df_step['mortality'], exog_11_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                      cluster_entity=True)
print(model_11_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.4500
Estimator:                   PanelOLS   R-squared (Between):             -1.7351
No. Observations:                3147   R-squared (Within):               0.3059
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.4263
Time:                        17:41:40   Log-likelihood                   -6378.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      202.86
Entities:                         137   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(12,2975)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             20.898
                            

# Model 12: adding hospiatal beds variable:

In [50]:
exog_12_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd','dm_high_aging_society',
                                      'labor_rate * is_oecd','internet','smoking','health_exp','fertility','hosp_beds']])
model_12_fe = PanelOLS(df_step['mortality'], exog_12_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered', cluster_entity=True)
print(model_12_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              mortality   R-squared:                        0.4534
Estimator:                   PanelOLS   R-squared (Between):             -1.4710
No. Observations:                3124   R-squared (Within):               0.3238
Date:                Fri, Mar 27 2026   R-squared (Overall):             -1.1930
Time:                        17:42:13   Log-likelihood                   -6289.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      188.38
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(13,2952)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             19.572
                            

/Users/maryammoradi/enter/envs/thesis/lib/python3.10/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


# Comparing results of all Fixed Effect models:

In [51]:
Compare_results_mortality_fix_Effect = compare({
    'FE model 1': model_1_fe,
    'FE model 2': model_2_fe,
    'FE model 3': model_3_fe,
    'FE model 4': model_4_fe,
    'FE model 5': model_5_fe,
    'FE model 6': model_6_fe,
    'FE model 7': model_7_fe,
    'FE model 8': model_8_fe,
    'FE model 9': model_9_fe,
    'FE model 10': model_10_fe,
    'FE model 11': model_11_fe,
    'FE model 12': model_12_fe,
    })

print(Compare_results_mortality_fix_Effect)

                                                                                              Model Comparison                                                                                             
                             FE model 1     FE model 2     FE model 3    FE model 4     FE model 5     FE model 6     FE model 7     FE model 8     FE model 9    FE model 10    FE model 11    FE model 12
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                 mortality      mortality      mortality     mortality      mortality      mortality      mortality      mortality      mortality      mortality      mortality      mortality
Estimator                      PanelOLS       PanelOLS       PanelOLS      PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS       Pane

In [30]:
with open('Compare_mortality_table_v2.html','w') as f:
    f.write(Compare_results_mortality_fix_Effect.summary.as_html ())